In [1]:
import pandas as pd
import joblib

from utils import fit_position_pipeline, select_features_by_correlation

In [ ]:
features = joblib.load('../predictors/hist/feats_fwd')
player_data = pd.read_csv('../rolled_data_24_25.csv')
player_data = player_data[player_data['position'] == 'Forward']

league_models = {}

for gw in range(6,39):
    print(f"\n{'=' * 60}\n Model for round {gw} \n{'=' * 60}")
    data = player_data[(player_data['round']>3) & (player_data['round'] < gw)].fillna(0)
    feats = features[gw]
    feats.remove('xP')
    selected_feats = select_features_by_correlation(data, feats, target_col='xP', redundancy_threshold=0.45, min_corr=0.15, verbose=True)


    league_model = fit_position_pipeline(
            data, selected_feats, gw, position_name='FWD', random_seed=42
        )

    print(f"Features for FWD kept in gw {gw} -> {league_model['feats']}")

    league_models[gw] = league_model



 Model for round 6 

Selected 17 / 17 features (ranked by |corr| with 'xP', redundancy_threshold=0.5, min_corr=0.1):
['opponent_difficulty', 'elo_diff', 'ownership_change', 'creativity_rolling_1', 'interceptions_rolling_3', 'blocks_rolling_1', 'tackles_won_percent_rolling_1', 'blocks_per_90_rolling_3', 'tackles_won_percent_per_90_rolling_3', 'assists_5_ewm', 'yellow_cards_5_ewm', 'clearances_3_ewm', 'opponent_xG_1_ewm', 'big_chances_faced_5_ewm', 'goals_scored_per_90_1_ewm', 'assists_per_90_1_ewm', 'expected_goal_involvements_per_90_3_ewm']

Position: FWD
n_players=146, n_features=17
X shape: (146, 17)
rank: 17  (full rank would be 17)
condition number: 3.61e+00
  (as a rough guide: >1e3 is concerning, >1e6 is basically singular)
Fitting initial model...


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


Output()

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 44 seconds.


  divergences: 0
  dropping 'assists_5_ewm' (label beta[9]) -- HDI near-duplicate of 'assists_per_90_1_ewm' (label beta[15])
  dropping 'yellow_cards_5_ewm' (label beta[10]) -- HDI near-duplicate of 'assists_per_90_1_ewm' (label beta[15])
  dropping 'creativity_rolling_1' (label beta[3]) -- HDI near-duplicate of 'tackles_won_percent_per_90_rolling_3' (label beta[8])
  dropping 'expected_goal_involvements_per_90_3_ewm' (label beta[16]) -- HDI near-duplicate of 'elo_diff' (label beta[1])
Kept 4 / 17 features: ['elo_diff', 'ownership_change', 'tackles_won_percent_per_90_rolling_3', 'assists_per_90_1_ewm']
Refitting on pruned feature set...


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


Output()

In [ ]:
joblib.dump(league_models, './estimates/league_models_fwd')